# ⚡ 新能源行业智能体
**一键部署到 Colab Free (T4 GPU) - Gradio 公网访问**

---

## ⚠️ 运行前必须配置 Secrets

**在 Colab 左侧边栏 → 🔑 钥匙图标 → Secrets → 添加以下密钥:**

| Secret 名称 | 值 | 说明 |
|------------|-----|------|
| `HF_TOKEN` | `hf_xxx...` | HuggingFace Token（可选）|
| `NGROK_TOKEN` | `xxx...` | ngrok Token（可选，https://ngrok.com 免费注册）|

---

## 💾 数据持久化说明

| 数据 | 存储位置 | 断开后 |
|------|---------|--------|
| 模型文件 (~2.5GB) | Google Drive `/hf_cache` | ✅ 保留 |
| pip 下载包 | Google Drive `/pip_cache` | ✅ 保留 |
| vLLM CUDA kernel | Google Drive `/vllm_cache` | ✅ 保留 |
| 电价缓存 DB | Google Drive `/new-energy-data` | ✅ 保留 |
| 代码 | GitHub | ✅ 每次自动重拉 |

> 📌 **断开重连后重跑所有 Cell，模型/缓存/内核都不丢，启动速度越来越快**

---

In [ ]:
# ── Cell 1: 读取 Colab Secrets ──
import os

try:
    from google.colab import userdata
    for secret_name in ["HF_TOKEN", "NGROK_TOKEN"]:
        try:
            val = userdata.get(secret_name)
            if val:
                os.environ[secret_name] = val
                print(f"✅ {secret_name} 已加载")
            else:
                print(f"⚠️ {secret_name} 未配置（可选）")
        except Exception:
            print(f"⚠️ {secret_name} 未配置（可选）")
except ImportError:
    print("⚠️ 非 Colab 环境，从环境变量读取")

print("✅ Secrets 加载完成")

In [ ]:
# ── Cell 2: 挂载 Google Drive + 创建持久化目录 ──
import os, shutil

# 如果 /content/drive 已有残留文件，先清理再挂载
drive_mount = "/content/drive"
if os.path.isdir(drive_mount) and os.listdir(drive_mount):
    # 检查是不是真的已挂载 (有 MyDrive 目录)
    if os.path.isdir(os.path.join(drive_mount, "MyDrive")):
        print("✅ Google Drive 已挂载，跳过")
    else:
        print("清理残留文件后重新挂载...")
        for item in os.listdir(drive_mount):
            item_path = os.path.join(drive_mount, item)
            try:
                if os.path.isdir(item_path):
                    shutil.rmtree(item_path, ignore_errors=True)
                else:
                    os.remove(item_path)
            except:
                pass
        from google.colab import drive
        drive.mount(drive_mount)
else:
    from google.colab import drive
    drive.mount(drive_mount)

# 创建所有持久化目录
dirs = {
    "model_cache": "/content/drive/MyDrive/models",
    "hf_cache": "/content/drive/MyDrive/hf_cache",
    "pip_cache": "/content/drive/MyDrive/pip_cache",
    "vllm_cache": "/content/drive/MyDrive/vllm_cache",
    "data_dir": "/content/drive/MyDrive/new-energy-data",
}
for name, path in dirs.items():
    os.makedirs(path, exist_ok=True)

os.environ["HF_HOME"] = dirs["hf_cache"]
os.environ["HF_HUB_CACHE"] = dirs["hf_cache"]
os.environ["PIP_CACHE_DIR"] = dirs["pip_cache"]
os.environ["VLLM_CACHE_DIR"] = dirs["vllm_cache"]
os.environ["VLLM_CONFIG_ROOT"] = dirs["vllm_cache"]
os.environ["NEW_ENERGY_DATA_DIR"] = dirs["data_dir"]

# 检查电价缓存
import sqlite3
db_path = os.path.join(dirs["data_dir"], "electricity_cache.db")
if os.path.exists(db_path):
    count = sqlite3.connect(db_path).execute("SELECT COUNT(*) FROM electricity_prices").fetchone()[0]
    print(f"📊 电价缓存: {count} 条记录 (从上次会话恢复)")
else:
    print("📊 电价缓存: 空 (首次运行)")

print(f"💾 所有缓存目录已在 Google Drive 就绪")
print(f"   模型:  {dirs['model_cache']}")
print(f"   数据:  {dirs['data_dir']}")

In [ ]:
# Cell 3: Install dependencies (cache on Google Drive)
import os, glob, subprocess

pip_cache = os.environ.get("PIP_CACHE_DIR", "/content/drive/MyDrive/pip_cache")
vllm_cache = os.environ.get("VLLM_CACHE_DIR", "/content/drive/MyDrive/vllm_cache")

kernels = glob.glob(os.path.join(vllm_cache, '*.so'))
if kernels:
    print(f"vLLM CUDA kernel cache: {len(kernels)} files")

# Detect CUDA version
print("Detecting CUDA version...")
cuda_ver = None
try:
    import re
    nvcc_out = subprocess.run(["nvcc","--version"], capture_output=True, text=True).stdout
    m = re.search(r'release (\d+\.\d+)', nvcc_out)
    if m:
        cuda_ver = m.group(1)
        print(f"   CUDA: {cuda_ver}")
except:
    pass

print(f"pip cache: {pip_cache}")
print("Installing packages...")

# First: ensure numpy 2.x (undo accidental downgrade from previous run)
!pip install -q "numpy>=2.0"

# vLLM 0.6.3+ for CUDA 12 (Colab T4). Latest 0.7+ needs CUDA 13.
if cuda_ver and cuda_ver.startswith('12'):
    print(f"   vLLM 0.6.x (CUDA {cuda_ver} compatible)")
    !pip install -q "vllm>=0.6.3,<0.7.0" gradio plotly pandas duckduckgo_search pyngrok huggingface_hub openai httpx requests
else:
    !pip install -q vllm gradio plotly pandas duckduckgo_search pyngrok huggingface_hub openai httpx requests

print("Done")


In [ ]:
# ── Cell 4: Clone 代码仓库 ──
import os

repo_dir = "/content/new-energy-agent"

if os.path.exists(repo_dir):
    print("仓库已存在，拉取最新代码...")
    %cd {repo_dir}
    !git pull
else:
    print("克隆仓库...")
    !git clone https://github.com/pai-pixel/new-energy-agent.git {repo_dir}
    %cd {repo_dir}

print(f"✅ 代码目录: {os.getcwd()}")
!ls -la src/

In [ ]:
# ── Cell 5: 下载/加载模型到 Google Drive ──
import os, time

# 模型缓存到 Drive (断开不丢)
HF_CACHE = "/content/drive/MyDrive/hf_cache"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct-AWQ"

os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_CACHE"] = HF_CACHE

# 检查模型是否已在 Drive 缓存
from huggingface_hub import snapshot_download, scan_cache_dir

def get_cached_path(repo_id):
    """检查模型是否已缓存，返回本地路径或 None"""
    try:
        cache_info = scan_cache_dir(cache_dir=HF_CACHE)
        for repo in cache_info.repos:
            if repo.repo_id == repo_id and repo.revisions:
                # 找最新的完整快照
                for rev in repo.revisions:
                    if not rev.last_modified:
                        continue
                    path = os.path.join(HF_CACHE, "hub", repo.repo_path, "snapshots", rev.commit_hash)
                    if os.path.isdir(path) and os.listdir(path):
                        return path
    except Exception:
        pass
    return None

cached_path = get_cached_path(MODEL_ID)

if cached_path:
    print(f"✅ 模型已缓存 (Google Drive)")
    print(f"   路径: {cached_path}")
    # 检查大小
    total_size = sum(
        os.path.getsize(os.path.join(dirpath, f))
        for dirpath, _, filenames in os.walk(cached_path)
        for f in filenames
    )
    print(f"   大小: {total_size / 1e9:.1f} GB")
else:
    print(f"📥 首次下载模型到 Google Drive...")
    print(f"   模型: {MODEL_ID}")
    print(f"   目标: {HF_CACHE}")
    print(f"   约 2-5 分钟，请耐心等待...")
    start = time.time()
    cached_path = snapshot_download(
        MODEL_ID,
        cache_dir=HF_CACHE,
        resume_download=True,
        max_workers=4,
    )
    elapsed = time.time() - start
    print(f"✅ 下载完成! 耗时 {elapsed:.0f} 秒")
    print(f"   路径: {cached_path}")

os.environ["MODEL_LOCAL_PATH"] = cached_path
print(f"\n💾 模型在 Google Drive，断开不丢失")
print(f"   HF_HOME={HF_CACHE}")

In [ ]:
# ── Cell 6: 验证 GPU + 启动 vLLM 推理服务 ──
import subprocess, os, time

# 1. 确认 GPU
print("🔍 检查 GPU...")
gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if not gpu_info:
    print("❌ 未检测到 GPU！运行时 → 更改运行时类型 → T4 GPU")
else:
    print(f"✅ GPU: {gpu_info[0]}")

# 2. 确认模型文件存在
model_path = os.environ.get("MODEL_LOCAL_PATH", "")
if not model_path or not os.path.isdir(model_path):
    print(f"\n❌ 模型路径无效: {model_path}")
    print("请重新运行 Cell 5 下载模型")
    raise SystemExit(1)

print(f"\n📂 模型路径: {model_path}")
print(f"   包含文件: {len(os.listdir(model_path))} 个")

# 3. 清理旧进程
!pkill -f "vllm.entrypoints" 2>/dev/null || true
time.sleep(2)

# 4. 启动 vLLM (读本地 Drive 路径，不走网络)
print(f"\n🚀 启动 vLLM 推理服务...")
print(f"   模型: {model_path}")
print(f"   端口: 8000")
print(f"   首次编译 CUDA kernel 约需 2-3 分钟...")

log_file = "/content/vllm.log"
with open(log_file, "w") as f:
    proc = subprocess.Popen([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--quantization", "awq",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--port", "8000",
        "--host", "0.0.0.0",
    ], stdout=f, stderr=f)

print(f"   PID: {proc.pid}")
print(f"   日志: tail -f {log_file}")
print(f"\n⏳ 等待 10 秒后运行 Cell 6.5 诊断...")
time.sleep(10)
print("👉 现在运行 Cell 6.5 检查 vLLM 状态")

In [ ]:
# ── Cell 6.5: 诊断 vLLM 启动状态 ──
# 运行此 Cell 查看 vLLM 是否正常启动

import time, os

print("=" * 50)
print("🔍 vLLM 启动诊断")
print("=" * 50)

# 1. 查看 vLLM 进程
print("\n📋 vLLM 进程状态:")
!ps aux | grep vllm | grep -v grep || echo "  ❌ 未找到 vLLM 进程"

# 2. 查看最新日志
print("\n📜 vLLM 最新日志 (最后 30 行):")
log_file = "/content/vllm.log"
if os.path.exists(log_file):
    !tail -30 {log_file}
else:
    print("  ❌ 日志文件 /content/vllm.log 不存在")

# 3. 检查端口
print("\n🔌 端口 8000:")
!ss -tlnp | grep 8000 || echo "  ❌ 端口 8000 未监听"

# 4. 测试 API
print("\n🌐 API 连通性测试:")
try:
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
    models = client.models.list()
    print(f"  ✅ vLLM 已就绪! 模型: {[m.id for m in models]}")
except Exception as e:
    print(f"  ⚠️ vLLM 尚未就绪: {str(e)[:120]}")

print("\n" + "=" * 50)
print("看到 ✅ 就继续运行 Cell 7")
print("看到 ❌ 就回到 Cell 6 重新启动 vLLM")
print("=" * 50)

In [ ]:
# ── Cell 7: 启动智能体 ──
import os, sys, time
import gradio as gr

%cd /content/new-energy-agent
sys.path.insert(0, "/content/new-energy-agent")

from src.agent import NewEnergyAgent, create_ui, wait_for_vllm, start_ngrok
from IPython.display import display, Javascript
import logging
logging.basicConfig(level=logging.WARNING)

# 1. Wait for vLLM
print("Waiting for vLLM...")
if not wait_for_vllm(max_retries=60, interval=5):
    print("vLLM timeout! Run Cell 6.5 to diagnose")
    print("Check: !tail -50 /content/vllm.log")
    raise SystemExit(1)
print("vLLM ready!")

# 2. ngrok
ngrok_url = start_ngrok(7860)

# 3. Agent
print("Starting agent...")
agent = NewEnergyAgent()
demo = create_ui(agent)

# 4. Show access info
print("=" * 60)
print("  New Energy Agent Ready!")
print("=" * 60)
if ngrok_url:
    print(f"  ngrok: {ngrok_url}")
    display(Javascript(f'window.open("{ngrok_url}", "_blank");'))
print("  Gradio URL: check cell output below for .gradio.live")
print("  Cell will keep running (normal)")
print("  Stop: Colab top stop button")

# 5. Launch Gradio (blocking)
demo.queue(max_size=32).launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    show_error=True,
    css=".gradio-container{max-width:900px!important}",
    theme=gr.themes.Soft(primary_hue="green"),
)


In [ ]:
# ── Cell 8 (可选): 保活脚本 + 缓存统计 ──
from IPython.display import display, Javascript

keep_alive_js = """
function ClickConnect() {
    console.log('Colab 保活心跳: ' + new Date());
    document.querySelector('colab-connect-button').click();
}
setInterval(ClickConnect, 60000);
"""
display(Javascript(keep_alive_js))

# 显示缓存统计
import os, sqlite3
db_path = os.path.join(os.environ.get("NEW_ENERGY_DATA_DIR", "/content/drive/MyDrive/new-energy-data"), "electricity_cache.db")
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    total = conn.execute("SELECT COUNT(*) FROM electricity_prices").fetchone()[0]
    provinces = conn.execute("SELECT DISTINCT province FROM electricity_prices").fetchall()
    conn.close()
    print(f"📊 电价缓存: {total} 条 | 覆盖省份: {', '.join(p[0] for p in provinces)}")

print("✅ 保活脚本已启动（每分钟心跳）")
print("💾 模型 + 电价缓存均在 Google Drive，断开不丢失")
print("⚠️ 免费版 Colab 通常运行 4-12 小时后自动断开")
print("   断开后重新运行所有 Cell 即可恢复，无需重新下载模型")

---
## 📖 使用说明

### 基本对话
- 「你好」→ 闲聊 · 「上海上网电价」→ 电价 · 「北京天气」→ 天气 · 「光伏政策」→ 联网搜索

### 多轮上下文（自动继承）
- 「上海上网电价」→ 查询上海上网电价
- 「江苏呢」→ 自动继承「上网电价」，查江苏
- 「工商业电价呢」→ 自动继承「江苏」，切换电价类型
- 「那天气呢」→ 自动继承「江苏」，切换天气

### 断开重连
每次 Colab 断开后重新运行时，**只需按顺序重跑所有 Cell**。模型和电价缓存都保留在 Google Drive，不需要重新下载。

---
> ⚡ 新能源行业智能助手 | [GitHub](https://github.com/pai-pixel/new-energy-agent)